In [1]:
import json

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
whole_dataset = pd.read_csv("whole-dataset.csv", dtype=str).fillna("")

In [3]:
len(whole_dataset)

104060

In [4]:
len(whole_dataset) / 3

34686.666666666664

In [5]:
whole_dataset.columns

Index(['Product Type', 'Food Product Group', 'Food Product Category',
       'Primary Food Product Category', 'Product Name', 'Basic Type',
       'Sub-Type 1', 'Sub-Type 2', 'Sub-Type 3', 'Flavor/Cut', 'Shape', 'Skin',
       'Seed/Bone', 'Processing', 'Cooked/Cleaned', 'WG/WGR',
       'Dietary Concern', 'Additives', 'Dietary Accommodation', 'Frozen',
       'Packaging', 'Commodity'],
      dtype='object')

In [6]:
whole_dataset["concat"] = whole_dataset.iloc[:, 5:].fillna("").agg(", ".join, axis=1).str.strip(", ").replace('(?:, )+', ', ', regex=True)

In [7]:
whole_dataset[whole_dataset["Product Name"] != whole_dataset["concat"]][["Product Name", "concat"]]

,Product Name,concat
52638,"additives, thickener","thickener, powder"
52702,"appetizer, quiche, cheese, egg","quiche, variety"
52704,apple,"apple, gala"
52705,apple,"apple, jonathan"
52706,apple,"apple, red delicious"
...,...,...
97948,"yogurt, greek, nonfat","yogurt, nonfat"
97949,"yogurt, greek, nonfat","yogurt, nonfat"
97975,"yogurt, low fat, ss","yogurt, low fat"
98016,"yogurt, parfait, flavored","yogurt, parfait"


In [8]:
system_message = (
    "Your job is to classify a food product's attributes as a JSON object with the following keys and values:\n\n" +
    "* \"Food Product Group\": which must be present and its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Food Product Group"].value_counts().index if x != "") + "\n" +
    "* \"Food Product Category\": which must be present and its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Food Product Category"].value_counts().index if x != "") + "\n" +
    "* \"Primary Food Product Category\": which must be present and its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Primary Food Product Category"].value_counts().index if x != "") + "\n" +
    "* \"Basic Type\": if it is present, it must have a value like " + ", ".join(json.dumps(x) for x in whole_dataset["Basic Type"].value_counts().iloc[:30].index if x != "") + "\n" +
    "* \"Sub-Type\": if it is present, it is a list of values like " + ", ".join(json.dumps(x) for x in whole_dataset["Sub-Type 1"].value_counts().iloc[:30].index if x != "") + "\n" +
    "* \"Flavor/Cut\": if present, its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Flavor/Cut"].value_counts().index if x != "") + "\n" +
    "* \"Shape\": if present, its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Shape"].value_counts().index if x != "") + "\n" +
    "* \"Skin\": if present, its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Skin"].value_counts().index if x != "") + "\n" +
    "* \"Seed/Bone\": if present, its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Seed/Bone"].value_counts().index if x != "") + "\n" +
    "* \"Processing\": if present, its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Processing"].value_counts().index if x != "") + "\n" +
    "* \"Cooked/Cleaned\": if present, its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Cooked/Cleaned"].value_counts().index if x != "") + "\n" +
    "* \"WG/WGR\": if present, its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["WG/WGR"].value_counts().index if x != "") + "\n" +
    "* \"Dietary Concern\": if present, its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Dietary Concern"].value_counts().index if x != "") + "\n" +
    "* \"Additives\": if present, its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Additives"].value_counts().index if x != "") + "\n" +
    "* \"Dietary Accommodation\": if present, its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Dietary Accommodation"].value_counts().index if x != "") + "\n" +
    "* \"Frozen\": if present, its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Frozen"].value_counts().index if x != "") + "\n" +
    "* \"Packaging\": if present, its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Packaging"].value_counts().index if x != "") + "\n" +
    "* \"Commodity\": if present, its value must be one of the following: " + ", ".join(json.dumps(x) for x in whole_dataset["Commodity"].value_counts().index if x != "")
)

print(system_message)

Your job is to classify a food product's attributes as a JSON object with the following keys and values:

* "Food Product Group": which must be present and its value must be one of the following: "Produce", "Condiments & Snacks", "Meat", "Bread, Grains & Legumes", "Meals", "Milk & Dairy", "Beverages", "Non-Food", "Seafood"
* "Food Product Category": which must be present and its value must be one of the following: "Condiments & Snacks", "Vegetables", "Meals", "Fruit", "Grain Products", "Beverages", "Non-Food", "Roots & Tubers", "Chicken", "Beef", "Cheese", "Pork", "Turkey, Other Poultry", "Milk & Dairy", "Yogurt", "Seafood", "Legumes", "Milk", "Eggs", "Tree Nuts & Seeds", "Rice", "Meat", "Butter", "Fish (Wild)", "Fish (Farm-Raised)", "Produce"
* "Primary Food Product Category": which must be present and its value must be one of the following: "Condiments & Snacks", "Vegetables", "Fruit", "Grain Products", "Beverages", "Non-Food", "Cheese", "Roots & Tubers", "Meals", "Beef", "Chicken", 

In [9]:
with open("system-message-try3.txt", "w") as file:
    file.write(system_message)

In [10]:
dataset_as_json = []
for _, row in whole_dataset.iterrows():
    response = {
        "Food Product Group": row["Food Product Group"],
        "Food Product Category": row["Food Product Category"],
        "Primary Food Product Category": row["Primary Food Product Category"],
    }
    if row["Basic Type"] != "":
        response["Basic Type"] = row["Basic Type"]
    if row["Sub-Type 1"] != "" or row["Sub-Type 2"] != "" or row["Sub-Type 3"] != "":
        response = response | {
        "Sub-Type": [x for x in [row["Sub-Type 1"], row["Sub-Type 2"], row["Sub-Type 3"]] if x != ""],
    }
    response = response | {k: row[k] for k in whole_dataset.columns[9:] if k != "concat" and row[k] != ""}
    dataset_as_json.append({"messages": [
        # {"role": "system", "content": system_message},
        {"role": "user", "content": row["Product Type"]},
        {"role": "assistant", "content": json.dumps(response)},
    ]})

In [11]:
rng = np.random.default_rng(12345)
permutation = rng.permutation(np.arange(len(whole_dataset)))

In [12]:
with open("training-data-try3.jsonl", "w") as file:
    for index in permutation[:30000]:
        file.write(json.dumps(dataset_as_json[index]) + "\n")

with open("validation-data-try3.jsonl", "w") as file:
    for index in permutation[30000:60000]:
        file.write(json.dumps(dataset_as_json[index]) + "\n")

with open("testing-data-try3.jsonl", "w") as file:
    for index in permutation[60000:]:
        file.write(json.dumps(dataset_as_json[index]) + "\n")